In [14]:
# =============================================================================
# 7006SCN- Machine Learning and Big Data
# Task 1 – Problem Definition and Dataset(10%) - CLASSIFICATION
#
# Student : <Bharath Vaderahalli Lokesha>
# Student ID : <16859171>
#
# Problem- BINARY CLASSIFICATION — predict the loan approval outcome
# (Approved vs Denied) derived from the `action_taken` field.
#
# Objective:
# Load and validate the HMDA dataset using Apache Spark.
# =============================================================================

In [11]:
# Mounting Google Drive

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [12]:



# ── Cell 0: Environment Setup ─────────────────────────────────────────────────
# Commit 1: feat(task1): SparkSession initialised
# Dataset: HMDA 2023 National Loan-Level Dataset (NOT Kaggle, NOT teaching)
# Source : https://ffiec.cfpb.gov/data-publication/snapshot-national-loan-level-dataset/2023
# Licence: HMDA Public Loan-Level Dataset (CFPB/FFIEC)

import os, sys, subprocess, time

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "pyspark==3.5.1", "psutil", "--quiet"],
    check=True
)

os.environ["JAVA_HOME"]      = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = sys.executable

DATA = "/content/drive/MyDrive/7006SCN/data/raw"
OUTPUT  = "/content/drive/MyDrive/7006SCN/outputs"
MODELS  = "/content/drive/MyDrive/7006SCN/models"


for p in [DATA, OUTPUT, MODELS]:
    os.makedirs(p, exist_ok=True)

import psutil

print(f"RAM: {psutil.virtual_memory().total/1e9:.1f} GB — environment ready ✓")




RAM: 13.6 GB — environment ready ✓


In [13]:
# ── Cell 1: SparkSession Initialisation (CW mandatory) ────────────────────────
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("7006SCN_HMDA2023_Task1_DataUnderstanding")
    .master("local[*]")
    .config("spark.driver.memory",          "10g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.driver.extraJavaOptions","-Xss4m")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version}  UI: {spark.sparkContext.uiWebUrl}")

Spark 3.5.1  UI: http://331c2d4e46bc:4040


In [15]:
# ── Cell 2: Dataset Verification + File-Size Validation (CW mandatory) ─────────
# Commit 2: feat(task1): HMDA 2023 dataset verified — 11.56M rows, 99+ columns, 4.39GB
# Source : HMDA 2023 National Loan-Level Dataset (CFPB/FFIEC)
# URL    : https://ffiec.cfpb.gov/data-publication/snapshot-national-loan-level-dataset/2023

import subprocess, os

CSV_PATH = f"{DATA}/My_Data.csv"

if os.path.exists(CSV_PATH):
    print(f"Dataset located ✓ : {CSV_PATH}")
else:
    raise FileNotFoundError(f"Dataset not found: {CSV_PATH}")

print("\n=== FILE SIZE VERIFICATION — screenshot for Task 1 ===")
print(subprocess.run(["ls", "-lh", DATA], capture_output=True, text=True).stdout)

sz = os.path.getsize(CSV_PATH) / 1e9

lc = int(
    subprocess.run(
        ["wc", "-l", CSV_PATH],
        capture_output=True,
        text=True
    ).stdout.strip().split()[0]
) - 1

print(f"File size : {sz:.2f} GB")
print(f"Data rows : {lc:,}")

assert sz >= 1.0, "Dataset must be at least 1 GB"
assert lc >= 10_000_000, "Dataset must contain at least 10 million rows"

print("Big Data compliance: PASS ✓ (11,564,178 rows, 4.39 GB, Official Government Dataset)")

Dataset located ✓ : /content/drive/MyDrive/7006SCN/data/raw/My_Data.csv

=== FILE SIZE VERIFICATION — screenshot for Task 1 ===
total 4.1G
-rw------- 1 root root 4.1G Jun 12  2025 My_Data.csv

File size : 4.38 GB
Data rows : 11,564,178
Big Data compliance: PASS ✓ (11,564,178 rows, 4.39 GB, Official Government Dataset)


In [16]:
# ── Cell 3: Schema + df.show(20) (CW mandatory screenshots) ───────────────────
# Commit 2: feat(task1): load HMDA dataset, inspect schema and preview records

from pyspark.sql import functions as F

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_PATH)
    .cache()
)

print("="*70)
print("HMDA 2023 DATASET SUMMARY")
print("="*70)
print(f"Total rows    : {df_raw.count():,}")
print(f"Total columns : {len(df_raw.columns)}")
print("="*70)

print("=== df.printSchema() — screenshot for Task 1 ===")
df_raw.printSchema()
print()

print("=== df.show(20) — screenshot for Task 1 ===")
df_raw.show(20, truncate=True)

print("<-- Screenshot BOTH and embed in Task 1 report")

HMDA 2023 DATASET SUMMARY
Total rows    : 11,564,178
Total columns : 99
=== df.printSchema() — screenshot for Task 1 ===
root
 |-- activity_year: integer (nullable = true)
 |-- lei: string (nullable = true)
 |-- derived_msa_md: integer (nullable = true)
 |-- state_code: string (nullable = true)
 |-- county_code: string (nullable = true)
 |-- census_tract: string (nullable = true)
 |-- conforming_loan_limit: string (nullable = true)
 |-- derived_loan_product_type: string (nullable = true)
 |-- derived_dwelling_category: string (nullable = true)
 |-- derived_ethnicity: string (nullable = true)
 |-- derived_race: string (nullable = true)
 |-- derived_sex: string (nullable = true)
 |-- action_taken: integer (nullable = true)
 |-- purchaser_type: integer (nullable = true)
 |-- preapproval: integer (nullable = true)
 |-- loan_type: integer (nullable = true)
 |-- loan_purpose: integer (nullable = true)
 |-- lien_status: integer (nullable = true)
 |-- reverse_mortgage: integer (nullable = true